<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [1]</a>'.</span>

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [1]:
import os
import sys
import subprocess

# 1. تحديد مسار المكاتب التي قمت بتنزيلها مسبقاً على القرص الصلب
OFFLINE_PACKAGES_PATH = "/home/jovyan/work/storage/packages"

# إضافة المسار إلى نظام بايثون لضمان قراءة المكاتب فور تثبيتها
if OFFLINE_PACKAGES_PATH not in sys.path:
    sys.path.insert(0, OFFLINE_PACKAGES_PATH)

# 2. تثبيت المكاتب أوفلاين بالكامل من المجلد الفعلي (دون استهلاك كيلوبايت واحد من الإنترنت)
print("⚡ جاري تفعيل وتثبيت مكاتب التعلم الآلي أوفلاين من الهارد ديسك...")
try:
    REQUIRED_PACKAGES = ["numpy", "pandas", "scikit-learn", "scipy", "joblib"]
    
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "--no-index",                             # منع الاتصال بالإنترنت نهائياً
        "--find-links", OFFLINE_PACKAGES_PATH     # قراءة الملفات المحلية (.whl) فقط
    ] + REQUIRED_PACKAGES, stdout=subprocess.DEVNULL) # إخفاء المخرجات الطويلة لترتيب النوت بوك
    
    print("🎉 تم تفعيل المكاتب بنجاح أوفلاين!")
except Exception as e:
    print(f"⚠️ تنبيه أثناء التثبيت: {e} (قد تكون مثبتة بالفعل)")

# 3. الآن نستدعي مكاتب التعلم الآلي بعد تفعيلها
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
import joblib

# 4. بناء جلسة Spark وجلب تعريف PostgreSQL تلقائياً
from pyspark.sql import SparkSession

print("⚙️ جاري تشغيل جلسة Spark وجلب تعريف الاتصال بقاعدة البيانات...")
spark = SparkSession.builder \
    .appName("SmartHome-ML-Engine") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .getOrCreate()

MODEL_PATH = "/home/jovyan/work/storage/energy_model.pkl"

def fetch_historical_data_with_spark():
    print("📥 جاري سحب البيانات من PostgreSQL عبر Spark...")
    
    jdbc_url = "jdbc:postgresql://smarthome-postgres:5432/smarthome_energy"
    connection_properties = {
        "user": "smarthome_user",
        "password": "smarthome_password",
        "driver": "org.postgresql.Driver"
    }
    
    # قراءة البيانات عبر Spark
    spark_df = spark.read.jdbc(
        url=jdbc_url, 
        table="spark_windowed_energy", 
        properties=connection_properties
    )
    
    # معالجة سريعة لاستخراج الساعة من نافذة الوقت
    spark_df.createOrReplaceTempView("energy_table")
    processed_spark_df = spark.sql("""
        SELECT HOUR(window_end) as hour, zone, device_type, avg_power_watts 
        FROM energy_table
    """)
    
    # تحويل البيانات إلى Pandas لبدء عملية التعلم الآلي
    return processed_spark_df.toPandas()

def train_predictive_model():
    try:
        df = fetch_historical_data_with_spark()
    except Exception as e:
        print(f"❌ فشل سحب البيانات عبر سبارك: {e}")
        return None
        
    if df.empty or len(df) < 5:
        print("⚠️ البيانات المتوفرة في قاعدة البيانات غير كافية لتدريب النموذج حالياً.")
        return None

    print(f"📊 تم جلب {len(df)} سجل. جاري تدريب نموذج الذكاء الاصطناعي...")
    
    # تحويل الميزات النصية لبيانات رقمية (One-Hot Encoding)
    df_encoded = pd.get_dummies(df, columns=['zone', 'device_type'])
    X = df_encoded.drop(columns=['avg_power_watts'])
    y = df_encoded['avg_power_watts']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    model = DecisionTreeRegressor(max_depth=5, random_state=42)
    model.fit(X_train, y_train)
    
    score = model.score(X_test, y_test)
    print(f"🎯 تم تدريب النموذج وحفظه بنجاح! دقة التنبؤ: {score * 100:.2f}%")
    
    # حفظ أسماء الأعمدة لضمان مطابقتها عند التنبؤ المستقبلي
    model.feature_names_ = list(X.columns)
    
    joblib.dump(model, MODEL_PATH)
    print(f"💾 تم حفظ ملف النموذج في: {MODEL_PATH}")
    return model

if __name__ == "__main__":
    train_predictive_model()

⚡ جاري تفعيل وتثبيت مكاتب التعلم الآلي أوفلاين من الهارد ديسك...


🎉 تم تفعيل المكاتب بنجاح أوفلاين!


ModuleNotFoundError: No module named 'pyspark'